**ISD-1020 · Master ISD (apprenticeship) · Université Paris-Saclay**

**Before any edit:** `File → Save a copy in Drive`.
Work only on **your** copy.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, torch

print("sklearn", sklearn.__version__, "| torch", torch.__version__)

# Course helpers — skip this

This cell is **not** an exercise: it downloads the plotting helpers and dataset
loaders if they are not already next to the notebook. You do not need to read
it. Continue below.


In [ ]:
# Environment setup (when run outside the repository, e.g. Google Colab)
import sys
import urllib.request
from pathlib import Path

_RAW_BASE = "https://raw.githubusercontent.com/stephane-rivaud/M2-ISD-ML-DL/main"

for _start in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_start / "pyproject.toml").is_file() and (_start / "common").is_dir():
        if str(_start) not in sys.path:
            sys.path.insert(0, str(_start))
        break

try:
    for module_path in ("common/__init__.py", "common/plots.py", "data/fetch.py"):
        local_file = Path(module_path)
        if local_file.is_file() or any(
            (Path(p) / module_path).is_file() for p in sys.path if p
        ):
            continue
        local_file.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{_RAW_BASE}/{module_path}", local_file)
except Exception:
    print(
        "Could not download the course helpers. Clone or download this repository so common/ and data/ sit next to this notebook."
    )


# Anomaly detection — is this machine going to break?

ISD-1020 · Monday 28 September 2026 · afternoon (Stéphane Rivaud) ·
PUIO B210.

Lab **⏱ 120 min** (sum of the markers below). Afternoon budget:
~140 min of lab minus **~20 min** of pretrained-model demo
(a separate notebook, not this one).

Thread of the module: **1. Problem · 2. Data · 3. Baseline · 4. Model · 5. Ablation ·
6. Error analysis → Recommendation**.
This morning: forecasting (éCO2mix). This afternoon: **anomalies** on a
machine temperature. Colab CPU only.

Honest arrival point: on *this* series, a rolling z-score is
**hard to beat** for a small autoencoder. The goal is not
that the network wins; it is to **choose a threshold** and quantify
false alarms.

In [ ]:
import random

from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from common.plots import plot_learning_curves
from data.fetch import NAB_FOCUS_SERIES, load_nab

SEED = 0
ROLL = 288  # 24 h of 5-minute samples
Z_K = 3.0
MERGE_GAP = 12  # 1 h: consecutive alarms closer than this are one event
WINDOW = 48  # 4 h windows for the autoencoder
HIDDEN = 16
LATENT = 8
EPOCHS = 12
BATCH_SIZE = 256
LR = 1e-3
AE_QUANTILE = 0.95
TRAIN_END = pd.Timestamp("2014-01-15")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("seed", SEED)
print("focus series:", NAB_FOCUS_SERIES)

## The problem — ⏱ ~8 min

**Numenta Anomaly Benchmark**, series
`machine_temperature_system_failure`: internal temperature of an
**industrial machine**, one point every **5 minutes**, winter
2013–2014. Four anomaly **windows** are labelled (not isolated
points).

**Decision.** Do we page on-call? A false negative = a missed
incident. A false positive = a technician disturbed for nothing.

Labels are used to **evaluate**, not to train a supervised
classifier. In production we do not have the windows in advance.

In pairs (30 s): if the detector cries five times a week, who
answers, and at what hour?

## Data — ⏱ ~10 min

`load_nab()` loads the 7 `realKnownCause` series (local cache, otherwise
Numenta's public CSVs). We filter the focus series.
12 timestamps are duplicated on 7 January (close values): we
keep the first.

In [ ]:
raw = load_nab()
print("nab shape:", raw.shape)
print("columns:", list(raw.columns))
print("series:", sorted(raw["series"].unique()))

series = (
    raw.loc[raw["series"] == NAB_FOCUS_SERIES]
    .sort_values("timestamp")
    .drop_duplicates("timestamp", keep="first")
    .reset_index(drop=True)
)
values = series["value"].to_numpy(dtype=float)
flag = series["is_anomaly"].to_numpy(dtype=bool)
times = series["timestamp"]
n_weeks = (times.max() - times.min()).total_seconds() / (7 * 86400)

print("focus shape:", series.shape)
print("span:", times.min(), "→", times.max())
print(f"days = {(times.max() - times.min()).total_seconds() / 86400:.2f}")
print(f"weeks = {n_weeks:.3f}")
print(f"anomaly points = {int(flag.sum())} / {len(flag)} = {flag.mean():.3%}")
print("median value =", float(np.median(values)))
series.head()

In [ ]:
def labelled_windows(mask):
    """Inclusive (start, end) index pairs of contiguous True runs."""
    mask = np.asarray(mask, dtype=bool)
    starts = np.where(np.diff(mask.astype(np.int8), prepend=0) == 1)[0]
    ends = np.where(np.diff(mask.astype(np.int8), append=0) == -1)[0]
    return list(zip(starts.tolist(), ends.tolist()))


def alarm_events(pred, merge_gap=MERGE_GAP):
    """Collapse consecutive alarms within ``merge_gap`` steps into events."""
    runs = labelled_windows(pred)
    if not runs:
        return []
    merged = [list(runs[0])]
    for start, end in runs[1:]:
        if start - merged[-1][1] <= merge_gap:
            merged[-1][1] = end
        else:
            merged.append([start, end])
    return [(a, b) for a, b in merged]


windows = labelled_windows(flag)
print(f"{len(windows)} labelled windows (each {windows[0][1] - windows[0][0] + 1} points):")
for i, (start, end) in enumerate(windows, start=1):
    sl = series.iloc[start : end + 1]
    print(
        f"  W{i}  {sl['timestamp'].iloc[0]} → {sl['timestamp'].iloc[-1]}  "
        f"min={sl['value'].min():.2f}  max={sl['value'].max():.2f}  "
        f"mean={sl['value'].mean():.2f}"
    )

fig, ax = plt.subplots(figsize=(10.0, 3.0))
ax.plot(times, values, color="#004E7D", lw=0.6)
for start, end in windows:
    ax.axvspan(times.iloc[start], times.iloc[end], color="#63003C", alpha=0.28)
ax.set_xlabel("Time")
ax.set_ylabel("Temperature")
ax.set_title("Focus series — labelled windows in plum")
plt.show()

Four windows of **47 h** each (567 points). W2 falls to ~2 (brutal
failure); W4 to ~26; W1 and W3 are more **lukewarm** (troughs around 48–51).
~10% of the *points* are in a window, but there are only **4
events**. A point-wise score is not incident
detection — next section.

Temporal split for the autoencoder: we train on points
**before** `2014-01-15` (W1 and W2 already past), **excluding** the
labelled windows. Labels are not a target: they filter
a corpus "considered normal". The z-score itself has no fit.

In [ ]:
train_ok = (times < TRAIN_END).to_numpy() & ~flag
print("train-end:", TRAIN_END)
print("points before TRAIN_END:", int((times < TRAIN_END).sum()))
print("normal train points:", int(train_ok.sum()))
print("windows starting after TRAIN_END:", sum(times.iloc[a] >= TRAIN_END for a, _ in windows))

## Baseline: rolling z-score — ⏱ ~15 min

Goal: a **point-wise anomaly score** whose reference is only the
recent past (~24 h, `ROLL` steps). The score must be **causal**: a
long failure must not rewrite the baseline used to judge it.
Flag when the reduced score is extreme relative to `Z_K`; leave the
warm-up (incomplete window) silent — no alarm there.

**Exercise.** Build that causal score on the temperature series and
the boolean alarm mask under the rule above.

**Interface (cells below).** Leave `z` (aligned score; leading gaps
allowed) and `z_alarm` (boolean, warm-up → False) in the namespace.

In [ ]:
# To complete
...

In [ ]:
fig, ax = plt.subplots(figsize=(10.0, 3.0))
ax.plot(times, z.abs(), color="#63003C", lw=0.5)
ax.axhline(Z_K, color="#004E7D", ls="--", lw=1.4, label=f"|z| = {Z_K:g}")
for start, end in windows:
    ax.axvspan(times.iloc[start], times.iloc[end], color="#63003C", alpha=0.22)
ax.set_ylim(0, 8)
ax.set_xlabel("Time")
ax.set_ylabel("|z|")
ax.set_title("Rolling reduced score (24 h, causal)")
ax.legend(frameon=False)
plt.show()

## Counting a detection — ⏱ ~12 min

**Convention of this notebook** (say it at the oral exam; others exist):

1. A labelled window is a **hit** if **at least one** alarm
   falls inside it. Ten alarms in the same window = **one** hit, not
   ten.
2. Consecutive alarms, or separated by $\le$ 1 h (`MERGE_GAP=12`
   5 min steps), form **one event**. An event that overlaps
   a labelled window is a true detection; otherwise it is a
   **false alarm**.
3. **Recall** = windows hit / 4. **Precision** = TP events /
   (TP + FP).

Point-wise *recall* (among the 2,268 labelled points) is
**not** the same number. We print it beside, to see the gap.

**Exercise.** Turn that convention into scores you can compare across
detectors: window-level hit rate, merged-event precision (and F1),
plus the point-wise pair printed beside them. The helpers already
defined above may help.

**Interface (cells below).** `window_metrics(pred, mask)` → a `dict`
with at least `hits`, `n_windows`, `n_windows_hit`, `recall`,
`precision`, `f1`, `tp_events`, `fp_events`, `point_precision`,
`point_recall`.

In [ ]:
# To complete
...

In [ ]:
def print_window_metrics(title, metrics):
    """Pretty-print window-level and point-wise scores."""
    print(title)
    print(f"  hits              {metrics['hits']}")
    print(f"  windows hit      {metrics['n_windows_hit']} / {metrics['n_windows']}")
    print(f"  recall           {metrics['recall']:.3f}")
    print(f"  precision        {metrics['precision']:.3f}")
    print(f"  F1               {metrics['f1']:.3f}")
    print(f"  events TP / FP   {metrics['tp_events']} / {metrics['fp_events']}")
    print(f"  FA per week      {metrics['fp_events'] / n_weeks:.2f}")
    print(
        f"  point prec/rec   {metrics['point_precision']:.3f} / "
        f"{metrics['point_recall']:.3f}"
    )


z_metrics = window_metrics(z_alarm, flag)
print_window_metrics(f"rolling |z| >= {Z_K:g}  (merge {MERGE_GAP} steps = 1 h)", z_metrics)

Reread the two lines `recall` and `point prec/rec`. Catching the
4 windows with a weak *point* recall is possible: we rang
**during** the incident, not on every 5 min step. Another
convention (each alarm counts, no 1 h merge) **changes the
numbers** — fast track.

## Autoencoder on windows — ⏱ ~12 min

Idea: a network learns to **reconstruct** short windows
(`WINDOW` steps ≈ 4 h) drawn from the normal corpus. At inference, a
window that the network reconstructs badly (high MSE) is suspect.

Goal: a **symmetric** bottleneck MLP in PyTorch — compress then
expand — sized by the constants `WINDOW`, `HIDDEN`, `LATENT`.
Reconstruct a **standardised** series (not pixels in [0, 1]), so the
last layer stays linear.

Training windows are entirely contained in
`train_ok`. The scaler (mean, standard deviation) is fitted on these points
only.

**Exercise.** Build that autoencoder in PyTorch.

**Interface (cells below).** `build_autoencoder(...)` → an
`nn.Module` that maps a window to a window of the same length.

In [ ]:
# To complete
...

## Reconstruction error — ⏱ ~10 min

A window is assigned the reconstruction MSE **at its
last instant** (we align the score on the present). The first
`WINDOW - 1` points have no score.

**Exercise.** Train the autoencoder on the normal windows prepared
below: Adam + MSE in PyTorch. Choose a validation split that says
something honest about **unseen later** windows (no leakage from
the future into the fit). Seed the run so the curve is
reproducible. Keep the train and val loss histories and plot them.

**Interface (cells below).** `train_autoencoder(model, X, ...)` →
`(train_losses, val_losses)`. After the cell, leave `autoencoder`,
`train_losses`, and `val_losses` defined.

In [ ]:
def make_windows(x, window, allowed):
    """Windows of length ``window`` whose points are all in ``allowed``."""
    x = np.asarray(x, dtype=np.float32)
    allowed = np.asarray(allowed, dtype=bool)
    rows = []
    ends = []
    for end in range(window - 1, len(x)):
        start = end - window + 1
        if not allowed[start : end + 1].all():
            continue
        rows.append(x[start : end + 1])
        ends.append(end)
    return np.stack(rows), np.asarray(ends, dtype=int)


mu = float(values[train_ok].mean())
sigma = float(values[train_ok].std()) or 1.0
scaled = ((values - mu) / sigma).astype(np.float32)
X_train, _ = make_windows(scaled, WINDOW, train_ok)
X_all, all_ends = make_windows(scaled, WINDOW, np.ones(len(scaled), dtype=bool))
print("train windows:", len(X_train), "| all windows:", len(X_all))
print(f"scaler mu={mu:.3f}  sigma={sigma:.3f}  (train-normal only)")

In [ ]:
# To complete
...

In [ ]:
def plot_score_timeline(times, score, threshold, windows, *, ylabel, title, thr_label):
    """Plot a timestamp-aligned score, a horizontal threshold, and labelled spans."""
    fig, ax = plt.subplots(figsize=(10.0, 3.0))
    ax.plot(times, score, color="#63003C", lw=0.5)
    ax.axhline(threshold, color="#004E7D", ls="--", lw=1.4, label=thr_label)
    for start, end in windows:
        ax.axvspan(times.iloc[start], times.iloc[end], color="#63003C", alpha=0.22)
    ax.set_xlabel("Time")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(frameon=False)
    plt.show()


def print_policy_comparison(policies, n_weeks):
    """Pretty-print a list of (name, window_metrics dict) on the whole series."""
    cmp_rows = []
    for name, metrics in policies:
        cmp_rows.append(
            {
                "policy": name,
                "recall": metrics["recall"],
                "precision": metrics["precision"],
                "f1": metrics["f1"],
                "windows_hit": metrics["n_windows_hit"],
                "fp_events": metrics["fp_events"],
                "fa_per_week": metrics["fp_events"] / n_weeks,
                "point_recall": metrics["point_recall"],
            }
        )
    comparison = pd.DataFrame(cmp_rows)
    print(comparison.round(3).to_string(index=False))
    print(f"n_weeks = {n_weeks:.3f}")
    for name, metrics in policies:
        print(f"{name} hits:", metrics["hits"])
    return comparison

## Choosing the threshold — ⏱ ~12 min

Two ways, **not equivalent**:

- **Quantile** (no labels): set the threshold from the reconstruction
  error distribution **on the normal train** (constant
  `AE_QUANTILE`). That is what we can do in production.
- **F1 sweep** (with labels): walk high quantiles of the error on
  the whole series and keep the max *window/event* F1. Diagnostic,
  not a deployment procedure: future labels are not there.

**Exercise.** Score every timestamp by how badly its window is
reconstructed (align on the window's last instant; warm-up stays
silent). Set a production-style threshold from train-only errors,
then run the label-aware F1 sweep for comparison. Plot the score
with the production threshold (`plot_score_timeline`). Then compare
four pager policies on the whole series — rolling $|z|\ge 3$,
$|z|\ge 5$, AE train-quantile, AE F1-max — and print the table
(`print_policy_comparison`).

**Interface (cells below).** `reconstruction_error(model, X)` and
`alarms_from_score(score, threshold)`.

In [ ]:
# To complete
...

The F1 sweep **sees the labels**. The train quantile does not. We will cite
both; we would **deploy** the second (or the z-score, which does not even
have a training corpus).

## False alarms — ⏱ ~8 min

How many times does on-call get up **per week**, outside labelled
windows? Read that column in the table you just printed (1 h merge).
Who answers: the line's maintenance on-call (nights included).
The four policies are on **the whole** series (~11.25 weeks).

## What would we put in production? — ⏱ ~8 min

Typical reading (to confront with *your* table): the z-score at $|z|\ge 3$
and the autoencoder at quantile 0.95 catch the **same** 4 windows.
The AE often has a little **fewer** false alarms, not another
job. Raising to $|z|\ge 5$ calms the pager and **misses** the two
lukewarm incidents (W1, W3).

**Recommendation.** Deploy the **rolling z-score** $|z|\ge 3$,
events merged at 1 h, as the first detector. On-call can
absorb on the order of **five** false alarms per week on *this*
machine; that is not zero, it is a shop-floor rhythm. Do **not**
replace that with the autoencoder: same recall, too thin a gain for
a training job, a scaler, drift of the "normal" corpus and
a second artefact to watch. Keep the AE in **shadow** (score
logged, no page) for a few weeks if we want
to compare. If the plant cannot live with ~5 pages/week: raise
the threshold **and** add a daily visual review, accepting
to miss W1/W3.

Missing data for what follows: what the operator did at each
alarm (true / false / already known). Without that feedback, we do not recalibrate
the threshold.

## Fast track — ⏱ ~10 min (optional)

Two ablations: the **counting convention**, then the autoencoder
**window length**. The numbers move; the business
fixes the convention.

In [ ]:
# ⚡ Fast track (optional) — for those who have already done ML
print("merge-gap ablation on |z| >= 3 (precision is the one that moves)")
pred = z_alarm
wins = labelled_windows(flag)
hits = [bool(pred[a : b + 1].any()) for a, b in wins]
recall_hit = float(np.mean(hits))
for gap in (0, 12, 72):
    runs = labelled_windows(pred)
    merged = []
    for start, end in runs:
        if merged and start - merged[-1][1] <= gap:
            merged[-1] = (merged[-1][0], end)
        else:
            merged.append((start, end))
    tp = fp = 0
    for start, end in merged:
        if any(not (end < ws or start > we) for ws, we in wins):
            tp += 1
        else:
            fp += 1
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    print(
        f"  gap={gap:2d} steps  events={len(merged):3d}  "
        f"recall={recall_hit:.2f}  prec={prec:.3f}  FA/week={fp / n_weeks:.2f}"
    )

In [ ]:
# ⚡ Fast track (optional) — for those who have already done ML
print("AE window-length ablation (same train corpus, 8 epochs, q=0.95)")
for win_len in (24, 48, 144):
    torch.manual_seed(SEED)
    X_tr_w, _ = make_windows(scaled, win_len, train_ok)
    X_all_w, ends_w = make_windows(scaled, win_len, np.ones(len(scaled), dtype=bool))
    model_w = build_autoencoder(window=win_len)
    train_autoencoder(model_w, X_tr_w, epochs=8)
    err_w = reconstruction_error(model_w, X_all_w)
    err_tr_w = reconstruction_error(model_w, X_tr_w)
    score_w = np.full(len(values), np.nan)
    score_w[ends_w] = err_w
    pred_w = alarms_from_score(score_w, float(np.quantile(err_tr_w, AE_QUANTILE)))
    metrics_w = window_metrics(pred_w, flag)
    print(
        f"  window={win_len:3d}  recall={metrics_w['recall']:.2f}  "
        f"prec={metrics_w['precision']:.3f}  "
        f"FA/week={metrics_w['fp_events'] / n_weeks:.2f}  "
        f"hits={metrics_w['hits']}"
    )

## Attributions

- **Numenta Anomaly Benchmark** (`realKnownCause`, series
  `machine_temperature_system_failure`) : [NAB](https://github.com/numenta/NAB),
  Numenta Inc., licence
  [MIT](https://github.com/numenta/NAB/blob/master/LICENSE.txt).
- **Training loop / `nn.Sequential`**: official
  PyTorch tutorials (*Deep Learning with PyTorch: A 60 Minute Blitz*), licence
  [BSD-3-Clause](https://github.com/pytorch/tutorials).

## To go further at home

1. NAB repository — data, windows, and the NAB metric (more generous
   than a binary hit):
   [numenta/NAB](https://github.com/numenta/NAB)
2. *Deep Learning with PyTorch: A 60 Minute Blitz* — `nn` and the loop:
   [tutorial](https://pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html)

## Empirical protocol — ⏱ ~15 min

The six cells below are the thread of the module (and of the oral exam).
Answer in prose, tied to *this* notebook.

## Problem

Answer in prose (not only code).

1. Which **business decision** should the model inform, and what is the **target**?
2. Which **metric** matches that decision, and why not accuracy alone?
3. How do you separate training and test **without leakage** (split unit, time, stratification)?

## Data

Answer in prose (not only code).

1. What are the variables, their types, and any notable **imbalances** or volumes?
2. Which columns must you **not** use (identifiers, leakages, target sub-flags)?
3. What do you know about **quality** (missing values, outliers, drift) and temporal or business coverage?

## Baseline

Answer in prose (not only code).

1. Which **naive baseline** (majority class, mean, seasonal) and what score does it get?
2. Why is this baseline the **honest floor**, and not a "deliberately weak" model?
3. Which score must you **beat** to justify a more complex model?

## Model

Answer in prose (not only code).

1. Which **model family** do you choose, and why (not "because it is deep")?
2. How do you train it (split, validation, hyperparameters, loss)?
3. Does the model **beat the baseline** on the chosen metric, on an uncontaminated split?

## Ablation

Answer in prose (not only code).

1. What happens if you remove a family of variables, a regulariser, or a block of the network?
2. Which choice (features, architecture, horizon, threshold) **actually changes** the score?
3. Does the gain justify the extra **complexity** relative to the baseline?

## Error analysis → Recommendation

Answer in prose (not only code).

1. Where does the model go wrong (**segments**, error types, horizon)?
2. Are these errors **costly** for the business, and what would you change in the data or the model?
3. What concrete **recommendation**: deploy, do not deploy, stay on the baseline, or collect this specific data?